# Going further with single-cell data: sub-clustering, trajectories and gene sets

**SupBioTech M2 practical · advanced session** · human skin wound healing · scRNA-seq, donor PWH27

In the first practical we went from a raw count matrix to annotated cell types (keratinocytes, fibroblasts, macrophages...). That was a *global* view. Real questions are usually more specific: **inside** a cell type, are there sub-populations? Do cells form a **continuum** (a differentiation path) rather than separate groups? And when a group of genes goes up, **what does it mean biologically**?

This notebook answers these three questions on the **keratinocytes** (the epidermal cells, the ones that must regrow to close the wound):

| Part | Method | The question we ask |
|---|---|---|
| A | **Sub-clustering** | Which states exist inside the keratinocytes? |
| B | **Trajectory / pseudotime** (PAGA, diffusion pseudotime) | Do these states form an ordered path, and which genes change along it? |
| C | **Gene set enrichment** (over-representation and GSEA) | What biological processes do the changing genes represent? |
| D | Take-home | What can we trust, what can we not? |

> **Prerequisite.** You must have run the first notebook (`Analysis_sc.ipynb`) up to the end: we start from its last checkpoint, `adata_sc_annotated_cell_state.h5ad`.

## How to read this notebook

The rhythm is the same as in the first practical (**Concept**, **Run**, **Break it**, **Question**, **Take-home**). A new point of attention: in this session we use methods that are **very easy to run and very easy to over-interpret**. A pretty UMAP, a smooth trajectory or a table of significant pathways always come out, whatever the data. So we will repeatedly ask: *how would I know if this result were an artefact?*

## Part 1 · Setup

> **Concept** · Same logic as before: import libraries, fix the random seed, define paths. Two new libraries: **gseapy** (gene set enrichment: a Python wrapper of Enrichr and GSEA) and `adjusted_rand_score` from **scikit-learn**, a number that measures how similar two clusterings are (we use it in Part A).
>
> The new folder `data/gene_sets` is a small local cache: gene set libraries are downloaded once and stored there (Part C).

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata as ad
import harmonypy
import gseapy as gp
from sklearn.metrics import adjusted_rand_score
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
SEED = 0
np.random.seed(SEED)
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(5, 4))
sns.set_style("whitegrid")

BASE = '/Users/sergio/Spatiotemporal_roadmap_of_human_skin_wound_healing'
SC_DATA = f'{BASE}/data/GSE241132_RAW'                   # raw data + published metadata (input)
OUT = f'{BASE}/data/processed_data/SupBioTech'           # checkpoints of the first notebook / our results
GENESET_DIR = f'{BASE}/data/gene_sets'                   # local cache of gene set libraries
os.makedirs(OUT, exist_ok=True); os.makedirs(GENESET_DIR, exist_ok=True)

## Part 2 · Loading only the keratinocytes

> **Concept · why do we *restart* from the raw counts?** In the first notebook, normalization, highly variable genes (HVG), PCA and neighbors were computed on **all** cell types together. The HVG list was therefore dominated by the genes that distinguish a fibroblast from a macrophage. Those genes say nothing about the *differences between keratinocytes*. If we simply re-clustered the existing PCA, we would be looking at keratinocytes through a lens built for another question.
>
> The rule is: **when you zoom into a subset, redo the pipeline on the subset, starting from the raw counts** (normalization → HVG → PCA → neighbors → clustering). Only the cell selection is inherited.
>
> Technically: the checkpoint is large (several GB in memory). We open it with `backed="r"` (the file stays on disk, nothing is loaded), select the keratinocytes, and bring **only those** into memory with `.to_memory()`. Then we build a fresh `AnnData` from the raw counts that we saved in `layers["counts"]` in the first notebook.

In [ ]:
big = sc.read_h5ad(f"{OUT}/adata_sc_annotated_cell_state.h5ad", backed="r")
is_kera = (big.obs["cell_type"] == "Keratinocytes").values
sub = big[is_kera].to_memory()
big.file.close()

# a clean object: raw counts in X, only the metadata we need
kera = ad.AnnData(X=sub.layers["counts"].copy(),
                  obs=sub.obs[["sample", "cell_type", "cell_state", "paper_barcode"]].copy(),
                  var=pd.DataFrame(index=sub.var_names))
del sub

# published annotation of the authors (finer than the one of notebook 1), matched by barcode
paper_meta = pd.read_csv(f"{SC_DATA}/GSE241132_cell_metadata.txt", sep="\t", index_col=0)
kera.obs["paper_state"] = kera.obs["paper_barcode"].map(paper_meta["newCellTypes"])

sc.pp.filter_genes(kera, min_cells=10)       # genes seen in fewer than 10 keratinocytes carry no information here
print(kera)
print(kera.obs["sample"].value_counts().sort_index().to_dict())
print(kera.obs["paper_state"].value_counts().to_dict())

Notice the last output: the authors distinguished several keratinocyte **states** (`Bas-I`, `Bas-II`, `Bas-mig`, `Bas-prolif`, `HF`, `Spi-I`, `Spi-II`, `Spi-mig`, `Gra-I`...: basal, hair follicle, spinous, granular, "mig" = migrating/wound-activated). We will **not** use these labels to build our clusters. We will use them at the end as an **external yardstick**: an independent opinion to compare with, not a ground truth.

## Part A · Sub-clustering the keratinocytes

**Goal.** Find the sub-populations hidden inside the keratinocytes, decide how many there are, and give them a biological name.

**To discuss together:**

- How many sub-clusters would you expect in the epidermis, and why?
- Is a cluster the same as a cell type? What is a *state*?
- How do we choose the resolution of the clustering, if there is no 'right' answer?

### A1 · Normalization, HVG, PCA, neighbors, UMAP

> **Concept** · Same five steps as notebook 1 (normalize to 10 000 counts, `log1p`, 2 000 HVG, 30 PCs, kNN graph, UMAP), now on 11 707 keratinocytes. One difference: the HVG are selected with `batch_key="sample"`, meaning "find the genes that are variable **within each sample**, then combine". This prevents a gene from being called variable only because it differs between D0 and D7 (a batch effect, or true biology, we cannot tell yet).
>
> We save the raw counts in `layers["counts"]` **before** normalizing, so that we can always go back.

In [ ]:
# Preprocess the keratinocytes: keep raw counts, normalize, log1p, 2000 HVG (per sample), PCA (30), neighbors, UMAP.
# Store this UMAP in kera.obsm['X_umap_uncorrected'].

<details>
<summary><b>Hint: keep the raw counts and the UMAP</b> (try first, open if stuck)</summary>

`normalize_total` and `log1p` overwrite `kera.X`: copy the counts to a layer first. Likewise `sc.tl.umap` overwrites `obsm['X_umap']`; make a copy under another name if you need two UMAPs.

```python
kera.layers["counts"] = kera.X.copy()
kera.obsm["X_umap_uncorrected"] = kera.obsm["X_umap"].copy()
```

</details>

*Why HVG per sample? (double-click to edit):* 

### A2 · What does the UMAP show? Biology or batch?

> **Concept** · We colour the same UMAP by three things: the **timepoint** (`sample`), the **authors' state**, and the **keratinocyte layer** from notebook 1. If cells separate by timepoint *within* a single state, then part of what we see is the wound itself (biology) and/or a technical batch effect. In a time course on one donor the two are confounded: D0, D1, D7, D30 are different biopsies processed separately.

In [ ]:
# UMAP (obsm key 'X_umap_uncorrected') coloured by sample, paper_state and cell_state.

*What do you see? Which groups are separated by timepoint? (double-click to edit):* 

### A3 · Clustering at three resolutions, without batch correction

> **Concept** · Leiden clustering has one main knob, the **resolution**: low = few large clusters, high = many small ones. There is no 'true' number. We try 0.3, 0.5, 0.8 and compare. We use the `igraph` implementation (`flavor="igraph", directed=False`), faster and the recommended one in scanpy.
>
> To measure whether a clustering is *driven by the batch* we compute the **sample purity**: for each cluster we take the fraction of cells coming from its most represented timepoint, and average over cells. With 4 timepoints, a clustering that ignores the timepoint has purity close to the overall share of the largest sample; a clustering that *is* the timepoint has purity 1.

In [ ]:
# Leiden clustering (igraph flavor) at resolutions 0.3, 0.5, 0.8 on the uncorrected graph. Store as unc_r0.3, unc_r0.5, unc_r0.8.

<details>
<summary><b>Hint: the Leiden flavor</b> (try first, open if stuck)</summary>

Without these two arguments scanpy warns and uses the old, slower backend.

```python
sc.tl.leiden(kera, resolution=r, key_added=f"unc_r{r}", flavor="igraph", directed=False, n_iterations=2, random_state=SEED)
```

</details>

In [ ]:
def sample_purity(labels, samples):
    t = pd.crosstab(labels, samples)
    return float(t.max(axis=1).sum() / t.values.sum())      # fraction of cells in the dominant sample of their cluster

def paper_ari(labels):
    ok = kera.obs["paper_state"].notna()
    return float(adjusted_rand_score(kera.obs.loc[ok, "paper_state"], labels[ok]))

def summarize(prefix, resolutions=(0.3, 0.5, 0.8)):
    rows = []
    for r in resolutions:
        lab = kera.obs[f"{prefix}_r{r}"]
        rows.append(dict(clustering=f"{prefix}_r{r}", n_clusters=lab.nunique(),
                         sample_purity=round(sample_purity(lab, kera.obs["sample"]), 3),
                         ARI_vs_paper=round(paper_ari(lab), 3)))
    return pd.DataFrame(rows)

summarize("unc")

> **Concept · the Adjusted Rand Index (ARI).** It compares two ways of grouping the same cells. ARI = 1: identical groupings; ARI ≈ 0: no more agreement than random labels. It does not require the clusters to have the same names or numbers, only to group the same cells. Here we compare *our* clusters with *the authors'* states (`ARI_vs_paper`).

In [ ]:
# Show the uncorrected clustering (unc_r0.5) and the sample on the UMAP, and a crosstab cluster x sample.

*How many clusters are dominated by a single timepoint? (double-click to edit):* 

### A4 · Batch correction *inside* the subset: Harmony on the timepoint

> **Concept** · In notebook 1 we had one donor and did not need any correction. Now, *within* the keratinocytes, the timepoint effect is strong enough to split states in two. **Harmony** (the method we met in notebook 1) moves the cells in PCA space so that each cluster contains cells of all samples, **without** touching the gene expression values. After Harmony we rebuild the graph and the UMAP from the corrected coordinates.
>
> Caution, this is a modelling choice with a cost: if a state *really* exists only at D7 (e.g. a wound-activated state), Harmony may partly merge it with its non-activated neighbours. That is why we will check afterwards that wound-related states are still visible (Part A6).

In [ ]:
# 1) Harmony on the PCA, batch variable = 'sample'; store the result in obsm['X_pca_harmony'].
# 2) New neighbors graph (key_added='harmony') + UMAP from it (store as X_umap_harmony).
# 3) Leiden at 0.3/0.5/0.8 on the harmony graph (keys har_r0.3...). 4) Compare with the uncorrected clusterings.

<details>
<summary><b>Hint: calling Harmony and the shape of its output</b> (try first, open if stuck)</summary>

`run_harmony` takes the PCA matrix, the `obs` table and the name(s) of the batch column(s). The corrected matrix is `Z_corr`, but depending on the version its shape is (PCs x cells) or (cells x PCs): check, and transpose if needed. To use it for the graph, pass `use_rep` and a separate `key_added` so that the first graph is not overwritten.

```python
ho = harmonypy.run_harmony(kera.obsm["X_pca"], kera.obs, ["sample"])
Z = np.asarray(ho.Z_corr)
if Z.shape[0] != kera.n_obs:
    Z = Z.T
kera.obsm["X_pca_harmony"] = Z
sc.pp.neighbors(kera, use_rep="X_pca_harmony", key_added="harmony", random_state=SEED)
```

</details>

*Compare the two tables: what changed in the purity and in the ARI? (double-click to edit):* 

In [ ]:
# Harmony UMAP (obsm 'X_umap_harmony') coloured by sample, har_r0.5 and paper_state.

*Do the timepoints now mix inside the clusters? (double-click to edit):* 

### A5 · Choosing the resolution and describing the clusters

> **Concept · there is no 'right' resolution.** Criteria to decide, in order of importance: (1) **biological interpretability**: can I name each cluster from its marker genes? (2) **stability**: do the clusters persist when I change the resolution slightly? (3) **agreement with an independent reference** (the ARI above). We take **resolution 0.5** with Harmony: enough clusters to separate the known epidermal layers and the wound-related states, few enough to be interpretable.
>
> To name the clusters we use two complementary tools:
>
> - **Differential expression** (`rank_genes_groups`, Wilcoxon test): the genes higher in a cluster than in all others.
> - **Marker signatures**: for the known biology of the epidermis we define short gene lists (basal, proliferating, spinous, granular, wound-activated, hair follicle) and compute a **score** per cell (`score_genes`: mean expression of the list minus the mean of a random set of genes with similar expression). Scores are less noisy than a single gene.

In [ ]:
# 1) kera.obs['kcluster'] = the harmony clustering at resolution 0.5.
# 2) Differential expression of each cluster vs the rest (Wilcoxon, key_added='rank_genes_k').
# 3) Define marker signatures and score them (sc.tl.score_genes); show the mean score per cluster as a heatmap.

<details>
<summary><b>Hint: signature scores</b> (try first, open if stuck)</summary>

`score_genes` needs genes that exist in the dataset: filter your lists with `var_names` first, otherwise it raises an error. Use a score name per signature so they do not overwrite each other.

```python
genes = [g for g in genes if g in kera.var_names]
sc.tl.score_genes(kera, genes, score_name=f"score_{name}", random_state=SEED)
```

</details>

*Which cluster looks basal? proliferating? wound-activated? (double-click to edit):* 

In [ ]:
# Dot plot of the signature genes per cluster (compute the dendrogram first!) and a table with the top 8 DE genes of each cluster.

<details>
<summary><b>Hint: dendrogram before dotplot</b> (try first, open if stuck)</summary>

`dotplot(..., dendrogram=True)` reads a dendrogram stored in the object; compute it first with the same `groupby`.

```python
sc.tl.dendrogram(kera, "kcluster", use_rep="X_pca_harmony")
```

</details>

*Write, for each cluster, the name you would give and the 2-3 genes that justify it (double-click to edit):* 

### A6 · Naming the clusters

> **Concept** · We read the heatmap, the dot plot and the DE table together and give each cluster a name. Two cautions:
>
> - **Names are hypotheses.** Our names combine the layer (basal / spinous / granular), the activity (proliferating, wound-activated) and, when the genes do not tell a clear story, a descriptive label (`..._ribosomal_high`, `Stress_or_mixed`).
> - **Not every cluster is a cell type.** A cluster dominated by ribosomal and mitochondrial genes, or by heat-shock/stress genes, may be a technical state (dissociation stress, low-quality cells) rather than a biological one. We keep such clusters but we flag them.
>
> **Cluster numbers are not stable across computers.** Leiden uses random numbers and floating-point operations; two machines can give the *same groups* with *different numbers*. The solution below therefore does **not** use cluster numbers: it lists the marker genes of each state (from our reference run) and assigns each of *your* clusters to the state with which its top DE genes overlap most (one-to-one, Hungarian algorithm). If your clusters differ from ours, some may stay `unassigned`: then you name them by hand. Always check the assignment against your own dot plot.

In [ ]:
# Build a dictionary {cluster number: name} from YOUR dot plot / DE table and store it in kera.obs['kstate'].
# Naming idea: layer (basal/spinous/granular) + activity (proliferating, wound-activated); flag technical-looking clusters.
# Then plot kstate on the harmony UMAP.

<details>
<summary><b>Hint: mapping numbers to names</b> (try first, open if stuck)</summary>

Cluster labels are strings (`'0'`, `'1'`, ...). Build the dictionary with **string keys**, use `.map()` and convert to `category` afterwards. Every cluster must have a name, otherwise the missing ones become `NaN` and later steps fail.

```python
kera.obs["kstate"] = kera.obs["kcluster"].map(assign).astype("category")
```

</details>

*Your table of names, with the genes that justify each one (double-click to edit):* 

### A7 · Validation against the published annotation, and where the states live in time

> **Concept** · We cross-tabulate **our states** (rows) against **the authors' states** (columns). Normalizing each row to 100% shows, for each of our states, which published state its cells belong to. A good annotation gives rows with one dominant column; a split row means our cluster merges (or splits) published states.

In [ ]:
# Row-normalized crosstab our kstate x paper_state, as a heatmap. Which of our states match one published state? Which are mixed?

*Where do our states agree and disagree with the authors? (double-click to edit):* 

> **Concept · composition over time.** Now the biological payoff: the proportion of each state **in each timepoint**. In a wound, we expect basal/proliferating states to be present at D0 (normal skin), the wound-activated states to expand early (D1, D7) and to return to baseline at D30.
>
> Caution: proportions are **compositional** (they sum to 100%): if one state grows, the others shrink even if their absolute number is constant. And with one donor and one biopsy per timepoint, we have **no replicates**: we can describe, we cannot test.

In [ ]:
# Stacked bar plot: % of each state in each timepoint.

*Which states change over time? Is it what you expected? (double-click to edit):* 

**Checkpoint.** We save the keratinocytes with their sub-states: Part B and C start from this file.

In [ ]:
kera.write_h5ad(f"{OUT}/keratinocytes_substates.h5ad")
print("saved:", f"{OUT}/keratinocytes_substates.h5ad")

## Part B · Trajectory and pseudotime

**Goal.** Test whether the keratinocyte states form an ordered path (basal → spinous → granular), assign every cell a position along it, and find the genes that change along the path.

**To discuss together:**

- Is differentiation a set of discrete boxes or a continuum? What would each view predict in the data?
- A trajectory is computed from a snapshot. What does 'time' mean here?
- What should we fix before computing a pseudotime, and who decides it: the algorithm or us?

### B1 · The idea of pseudotime

> **Concept** · Each cell is a **snapshot**: we destroyed it to read its RNA. But in a tissue that is renewing itself, cells are at *different moments* of the same process (a basal cell, a cell leaving the basal layer, a cell about to cornify...). If the process is continuous, the cells form a **line (or tree) in gene-expression space**, and we can order them along it. The position along that line is the **pseudotime**.
>
> Three crucial points:
>
> 1. **Pseudotime is not real time.** It is a measure of *transcriptional progress* along a path. Here it will represent *epidermal differentiation*, not the days after wounding (D0 → D30).
> 2. **The direction is not computed, it is decided.** The algorithm knows distances between cells, not which end is the 'start'. We must choose a **root cell** from biological knowledge (basal cells are the stem-like layer).
> 3. **A trajectory always comes out.** Give the algorithm any cloud of points and it will draw a path through it. Only biology tells us whether it is meaningful.
>
> We use two tools: **PAGA** (a coarse "map" of which states are connected) and **diffusion pseudotime, DPT** (a number per cell, computed from random walks on the kNN graph).

### B2 · PAGA: a map of connectivity between states

> **Concept** · PAGA (*partition-based graph abstraction*) takes the clusters (here our states) and asks, for each pair, *how many kNN edges connect their cells compared with what is expected by chance?* The result is a small graph: **nodes = states**, **edges = connectivity** (thicker = stronger). It is robust to the noisy detail of UMAP and it distinguishes a continuum (connected chain) from truly separate populations (disconnected nodes).
>
> We compute it on the **Harmony graph** (`neighbors_key="harmony"`), consistently with how we defined the states.

In [ ]:
# PAGA between the states (groups='kstate') on the harmony graph, then plot it (threshold ~0.05).

<details>
<summary><b>Hint: which graph?</b> (try first, open if stuck)</summary>

`sc.tl.paga` uses the default `neighbors` graph unless you tell it otherwise. Our states were defined on the Harmony graph, stored under the key `harmony`.

```python
sc.tl.paga(kera, groups="kstate", neighbors_key="harmony")
```

</details>

*Which states are strongly connected? Which are isolated? (double-click to edit):* 

### B3 · Choosing the root and computing diffusion pseudotime

> **Concept · diffusion pseudotime (DPT).** From the root cell, imagine a random walker that moves on the kNN graph, preferring close neighbours. The time the walker needs to reach each cell, averaged over all possible walk lengths, is that cell's pseudotime: close to the root = small, far = large. Technically scanpy first computes a **diffusion map** (a non-linear low-dimensional embedding where distances reflect walk probabilities) and then DPT on it.
>
> **The root.** We pick it in a data-driven but biologically justified way: the state with the highest **basal signature score**, and inside it the cell with the highest score. `kera.uns["iroot"]` must be the **integer position** of that cell in the object (not its name).

In [ ]:
# 1) Root = the cell with the highest 'score_Basal' within the cluster with the highest mean basal score. kera.uns['iroot'] = INTEGER position.
# 2) Diffusion map, then DPT (both on the 'harmony' graph). Keep a copy of the result in obs['pt_basal_root'].

<details>
<summary><b>Hint: the root must be an integer, and use the right graph</b> (try first, open if stuck)</summary>

`iroot` is the row **number** of the cell (`np.where(...)` gives positions, not names). `diffmap` and `dpt` also take `neighbors_key`: if you forget it they silently use the *uncorrected* graph.

```python
kera.uns["iroot"] = int(cand[np.argmax(basal[cand])])
sc.tl.diffmap(kera, neighbors_key="harmony")
sc.tl.dpt(kera, neighbors_key="harmony")
```

</details>

*Is the root state the one you expected? (double-click to edit):* 

In [ ]:
# Pseudotime, state and timepoint on the harmony UMAP; then boxplots of pseudotime per state (ordered by median) and per timepoint.

*Does the order of the states along pseudotime agree with epidermal differentiation? Does pseudotime correlate with the timepoint? (double-click to edit):* 

### B4 · Break it: what if we choose another root?

> **Break it** · The algorithm does not know the biology. We now put the root on the most **granular** cell (the terminal layer) and recompute. If the data really form a basal-to-granular line, the new pseudotime should be the *mirror image* of the first one (strongly negative rank correlation). Different results would reveal that the structure is not a simple line.

In [ ]:
# Recompute DPT with a root in the most granular cell, compare with the basal-root pseudotime (Spearman + scatter),
# and restore the basal root and pseudotime afterwards.

*What does the correlation (and the shape of the scatter) tell you? (double-click to edit):* 

### B5 · Which genes change along the path?

> **Concept** · The point of pseudotime is to find **genes whose expression varies along it**. A simple, robust approach: for each gene compute the **Spearman correlation** between its expression and the pseudotime (rank-based, so insensitive to the exact scale and to outliers). Positive = increases along the path (late/differentiated), negative = decreases (early/basal).
>
> To avoid noise we keep only the genes detected in at least 10% of the cells. Then we draw the top genes as a **heatmap over pseudotime**: cells are sorted by pseudotime, grouped into 20 equal-size bins, the mean expression per bin is computed and z-scored per gene.
>
> Caution: **the p-values of such a test are meaningless here.** The pseudotime was computed *from* the expression of the same genes, so we are testing the genes against a quantity built from them (circularity), and 11 000 cells are not 11 000 independent samples. Use the ranking, not the p-value.

In [ ]:
# Keep genes expressed in >=10% of cells; Spearman correlation of each gene with the pseudotime (finite values only).
# Print the 12 most negative and the 12 most positive genes.

<details>
<summary><b>Hint: Spearman on thousands of genes, fast</b> (try first, open if stuck)</summary>

Looping `scipy.stats.spearmanr` over thousands of genes is slow. Spearman is the Pearson correlation of the **ranks**: rank the columns of a DataFrame, rank the pseudotime and use `corrwith`.

```python
rho = Xd.rank().corrwith(pd.Series(pt[ok]).rank())
```

</details>

*Do the top genes make biological sense (basal keratins at the start, cornified-envelope genes at the end)? (double-click to edit):* 

In [ ]:
# Heatmap over pseudotime: top 15 negative + top 15 positive genes, cells in 20 equal-size pseudotime bins, z-score per gene.
# Then the UMAP of 4 genes of your choice.

<details>
<summary><b>Hint: equal-size bins</b> (try first, open if stuck)</summary>

`pd.qcut` on the **ranks** of the pseudotime gives bins with the same number of cells (ties in the raw values can break `qcut`).

```python
bins = pd.qcut(pd.Series(pt[ok]).rank(method="first"), 20, labels=False)
```

</details>

*Which genes switch on/off earliest? Are there genes with a transient peak? (double-click to edit):* 

## Part C · Gene set enrichment

**Goal.** Turn a list of genes (or a ranking of all genes) into biological processes, using two complementary methods, and learn when each is appropriate.

**To discuss together:**

- A list of 150 genes is not a result. How do we get from genes to a biological statement?
- What is the 'universe' (background) of a test, and why does it change the answer?
- Why can the same data give different answers with a gene list (ORA) and a ranking (GSEA)?

### C1 · Gene sets: where do they come from?

> **Concept** · A **gene set** is a named list of genes that share a property: a biological process (Gene Ontology, **GO**), a pathway (Reactome, KEGG), a cell-type signature, a chromosomal location... A **library** is a collection of such sets. Libraries are curated by humans and by algorithms: they reflect what was *studied*, not what exists. Well-studied processes (cell cycle, inflammation) have large, rich sets; poorly studied tissues are under-represented. Always ask which library you used and which version.
>
> We use **GO Biological Process 2023** as distributed by Enrichr, through `gseapy`. A library is downloaded once and then cached as a local JSON file in `data/gene_sets`: this way the practical works again without internet and the result is reproducible (the library version does not change under our feet).

In [ ]:
# Write a function load_gene_sets(name) that reads the library from a local JSON cache in GENESET_DIR if present,
# otherwise downloads it with gseapy (gp.get_library) and saves it. Load GO_Biological_Process_2023 and look at the set sizes.

<details>
<summary><b>Hint: downloading a library</b> (try first, open if stuck)</summary>

`gp.get_library` returns a dictionary `{term: [genes]}`: it is JSON-friendly, so you can dump it as it is.

```python
lib = gp.get_library(name=name, organism="Human")
```

</details>

*Why is a local copy of the library better than downloading it every time? (double-click to edit):* 

### C2 · Method 1 · Over-representation analysis (ORA)

> **Concept** · ORA starts from a **list of genes** (e.g. the genes up-regulated in a cluster) and asks, for each gene set: *is the overlap between my list and the set larger than expected by chance?* It is a **hypergeometric test** (equivalently Fisher's exact test): drawing 150 genes at random from a universe of N genes, how likely is it to get k or more from a set of size K?
>
> Three decisions determine the result:
>
> 1. **The list.** Here: the top 150 genes with adjusted p < 0.05 and log fold change > 1 in the *wound-activated* keratinocyte cluster (the one with the highest `Wound_activated` score), versus all the other keratinocytes. A threshold is arbitrary: a different cut-off gives a different list.
> 2. **The universe (background).** Only genes that *could* have been in the list should count. In a single-cell experiment those are the genes that are expressed in our cells. A default background of "all genes in the library" would make *any* expressed-in-skin gene look enriched. We use the genes detected in our keratinocytes.
> 3. **Multiple testing.** We test thousands of sets: we use the **adjusted p-value** (Benjamini-Hochberg FDR), never the raw one.

In [ ]:
# 1) Target = the cluster with the highest mean 'score_Wound_activated'. 2) Gene list = top 150 genes (padj<0.05, logFC>1) from rank_genes_k.
# 3) Universe = the genes of kera. 4) ORA with gp.enrichr using your local gene_sets and the explicit background. Show the top 10 terms.

<details>
<summary><b>Hint: enrichr with a local library and a custom background</b> (try first, open if stuck)</summary>

Give `gene_sets` the dictionary (not a library name) and `background` a list of gene symbols: nothing is downloaded and the test uses *your* universe.

```python
enr = gp.enrichr(gene_list=up, gene_sets=gene_sets, background=background, outdir=None, no_plot=True)
```

</details>

*Do the top terms match what you know about wound-activated keratinocytes? (double-click to edit):* 

In [ ]:
# Bar plot of the top 10 terms (-log10 adjusted p-value).

*Is any term redundant with another (same genes under different names)? (double-click to edit):* 

### C3 · Break it: change the universe

> **Break it** · We repeat the test **without** declaring the background. `gseapy` then uses, as the universe, the genes that appear in the gene set library. Compare the adjusted p-values and the order of the terms with the previous table. This is a very common mistake (it is the default in many web tools), and it silently changes the p-values (and sometimes the order of the terms), because the test is run against a universe that does not describe our experiment.

In [ ]:
# Repeat the ORA without the 'background' argument and put the two adjusted p-values side by side.

*What changed? Which version would you trust and why? (double-click to edit):* 

### C4 · Method 2 · GSEA on a ranked list (preranked)

> **Concept** · ORA throws away information: it cuts the genes into 'in the list' / 'not in the list' and ignores how strong the change is. **GSEA** (gene set enrichment analysis) avoids the cut-off. It starts from **all genes ranked** by a statistic (here the Wilcoxon score of the comparison) and walks down the list, increasing a running sum when we meet a gene of the set and decreasing it otherwise. The maximum deviation is the **enrichment score (ES)**; it is normalised for set size (**NES**) and its significance is estimated by **permutations** (here of the gene labels, 200 for speed; use 1 000+ in a real analysis).
>
> NES > 0: the set is concentrated at the *top* of the ranking (up-regulated); NES < 0: at the *bottom* (down-regulated).
>
> The question we ask now is different from C2: **how does the whole keratinocyte population change 1 day after wounding (D1) compared with healthy skin (D0)?** The ranking is from `rank_genes_groups(sample, D1 vs D0)`.
>
> **Statistical caveat (important).** One donor, one biopsy per timepoint: the cells of D1 are not independent replicates of "wounded skin", they are 4 795 cells of the same biopsy. Differences between *biopsies* (position, depth, handling) are mixed with the effect of the wound. The ranking and the pathways are a hypothesis-generating description; confirming them would require several donors and a **pseudobulk** analysis (summing the counts per donor and timepoint).

In [ ]:
# 1) Differential expression D1 vs D0 on all keratinocytes (Wilcoxon, key 'rank_genes_d1').
# 2) Ranking = a pandas Series gene -> score, sorted decreasing. 3) gp.prerank with min_size=15, max_size=500, 200 permutations, seed=SEED.
# 4) Show the terms with the highest NES.

<details>
<summary><b>Hint: what prerank wants as input</b> (try first, open if stuck)</summary>

`rnk` must be a Series (index = gene symbols, values = score) **sorted in decreasing order**. `min_size`/`max_size` remove tiny and huge gene sets. Fix the `seed`, otherwise the permutations change from run to run.

```python
rnk = d1.set_index("names")["scores"].sort_values(ascending=False)
pre = gp.prerank(rnk=rnk, gene_sets=gene_sets, min_size=15, max_size=500, permutation_num=200,
```

</details>

*Which processes go up at D1? Which go down? (a warning about duplicated ranking values is normal: ties among genes detected in few cells) (double-click to edit):* 

In [ ]:
# Bar plot of the 8 most positive and 8 most negative NES, then the classic GSEA plot of the top term (gp.gseaplot).

*Read the GSEA plot: where are the genes of the set in the ranking (the black ticks)? (double-click to edit):* 

## Part D · Take-home messages and limits

## Deliverable

Prepare a short report (1-2 pages, figures included) with:

1. the table comparing uncorrected and Harmony clustering (purity, ARI) and **one sentence** on the choice of the correction;
2. the UMAP of your keratinocyte states with the **name and the 2-3 genes** that justify each one;
3. the pseudotime figure (UMAP + state boxplot) and the heatmap of the genes along it, with a comment on **two genes** you did not expect;
4. one ORA figure and one GSEA figure, with the answer to: *do they say the same thing, and what is the limit of each?*;
5. a paragraph "what I would not claim from these data" (at least three points).